# Ch 08 · Resets, State, Mocap and Control Loops

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/08-resets-state-and-control.ipynb)

Companion notebook for the chapter page `chapters/08-resets-state-and-control.html` ([page](../chapters/08-resets-state-and-control.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Rendering notebooks only: let glvnd find the NVIDIA EGL driver (from the official
# mujoco_warp tutorial). Must run before mujoco is imported by the setup cell below.
import os
os.environ.setdefault("MUJOCO_GL", "egl")
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
try:
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version": "1.0.0", '
                    '"ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
    print("EGL ICD file:", NVIDIA_ICD_CONFIG_PATH, "| MUJOCO_GL =", os.environ["MUJOCO_GL"])
except OSError as e:
    print("could not write the EGL ICD file (fine outside Colab):", e)

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
os.environ.setdefault("MUJOCO_GL", "egl")
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## 1. reset_data: all worlds or just some
`mjw.reset_data(m, d, reset=mask)` resets the worlds where the bool/int mask of shape `(nworld,)` is set: `qpos <- qpos0`, mocap poses from the model, `eq_active <- eq_active0`, and time, qvel, ctrl, act, `qacc_warmstart`, applied forces and sensordata zeroed. Other worlds are not touched.

In [ ]:
# Example 8.1 - ex01_masked_reset.py
"""Example 8.1 - reset some worlds and leave the others running (reset_data with a mask).

Four PLAYGROUND worlds take 5 steps, then worlds 0 and 2 are reset with a per-world bool mask.
Run: python ex01_masked_reset.py [--dump-json PATH]
  --dump-json PATH writes per-step time and ball height of every world (Rule D, reset-mask figure).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()  # parse_known_args: also works inside Jupyter

NWORLD = 4  # the mask below is written for 4 worlds; the lesson does not need more
NSTEP = 5

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

frames = []


def record():
    frames.append([[round(float(t), 3), round(float(z), 3)]
                   for t, z in zip(d.time.numpy(), d.qpos.numpy()[:, 2])])


record()
for _ in range(NSTEP):
    mjw.step(m, d)
    record()
print("before reset: time  =", d.time.numpy())
print("before reset: ball z =", d.qpos.numpy()[:, 2])

mask = wp.array([True, False, True, False], dtype=bool)  # shape (nworld,), bool or int
mjw.reset_data(m, d, reset=mask)
record()
print("after reset:  time  =", d.time.numpy())
print("after reset:  ball z =", d.qpos.numpy()[:, 2])

for _ in range(NSTEP):
    mjw.step(m, d)
    record()
print(f"{NSTEP} more steps: time =", d.time.numpy())

try:
    mjw.reset_data(m, d, reset=wp.array([True, False], dtype=bool))
except ValueError as e:
    print("wrong mask shape ->", f"ValueError: {e}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "dt": float(mjm.opt.timestep), "kind": "tiles",
                   "labels": ["time", "ball z"], "reset_frame": NSTEP + 1,
                   "mask": [1, 0, 1, 0], "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 2. reset_data_keyframe: a different start per world
An `int` key resets all worlds (and must be in `[0, nkey)`); an int `wp.array` gives one key per world, and out-of-range ids such as `-1` leave that world untouched.

In [ ]:
# Example 8.2 - ex02_keyframe_reset.py
"""Example 8.2 - reset worlds to a keyframe: one id for all worlds, or one id per world (-1 = skip).

PLAYGROUND has one keyframe "up" (ball and box lifted to z = 2, hinge at 0.3 rad).
Run: python ex02_keyframe_reset.py [--dump-json PATH]
  --dump-json PATH writes per-step time and ball height of every world (Rule D, keyframe mode).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 4
NSTEP = 5

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
print("nkey:", mjm.nkey, "| key 0 name:", mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_KEY, 0))

frames = []


def record():
    frames.append([[round(float(t), 3), round(float(z), 3)]
                   for t, z in zip(d.time.numpy(), d.qpos.numpy()[:, 2])])


# 1. one int: every world goes to keyframe 0
mjw.reset_data_keyframe(m, d, 0)
print("key 0 for all:      qpos[:, 2] =", d.qpos.numpy()[:, 2], "hinge =", d.qpos.numpy()[:, 14])
record()

# 2. let the worlds fall a little, then a per-world array: -1 (or any id outside [0, nkey)) skips
for _ in range(NSTEP):
    mjw.step(m, d)
    record()
print(f"after {NSTEP} steps:      qpos[:, 2] =", d.qpos.numpy()[:, 2])
keys = wp.array([0, -1, 0, -1], dtype=int)
mjw.reset_data_keyframe(m, d, keys)
record()
print("keys [0,-1,0,-1]:   qpos[:, 2] =", d.qpos.numpy()[:, 2], "time =", d.time.numpy())
for _ in range(NSTEP):
    mjw.step(m, d)
    record()

# 3. a plain int outside [0, nkey) is an error (only arrays may skip worlds)
try:
    mjw.reset_data_keyframe(m, d, 1)
except ValueError as e:
    print("int key 1 ->", f"ValueError: {e}")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "dt": float(mjm.opt.timestep), "kind": "tiles",
                   "labels": ["time", "ball z"], "reset_frame": NSTEP + 1,
                   "keys": [0, -1, 0, -1], "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 3. A reset does not recompute kinematics: call forward
Derived quantities (`xpos`, sensors, contacts) stay stale until `mjw.forward` or the next `mjw.step`. Note also that after a step `xpos` describes the start of that step.

In [ ]:
# Example 8.3 - ex03_forward_after_reset.py
"""Example 8.3 - a reset writes the state, not the things computed from it: call forward.

After reset_data_keyframe, qpos holds the keyframe but xpos and the sensors are stale until
mjw.forward (or the next mjw.step) recomputes them.
Run: python ex03_forward_after_reset.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

NWORLD = 2

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
ball = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "ball")

for _ in range(50):  # let the ball fall for 0.5 s
    mjw.step(m, d)
print("after 50 steps:  qpos z =", d.qpos.numpy()[0, 2], "| xpos z =", d.xpos.numpy()[0, ball, 2],
      "| hinge sensor =", d.sensordata.numpy()[0, 0])

mjw.reset_data_keyframe(m, d, 0)  # key "up": ball at z = 2, hinge at 0.3 rad
print("after reset:     qpos z =", d.qpos.numpy()[0, 2], "| xpos z =", d.xpos.numpy()[0, ball, 2],
      "| hinge sensor =", d.sensordata.numpy()[0, 0], " <- xpos stale, sensordata zeroed")

mjw.forward(m, d)  # position, velocity, acceleration stages; no integration
print("after forward:   qpos z =", d.qpos.numpy()[0, 2], "| xpos z =", d.xpos.numpy()[0, ball, 2],
      "| hinge sensor =", d.sensordata.numpy()[0, 0])
print("time after forward:", d.time.numpy(), "(forward does not advance time)")

## 4. Snapshots: get_state / set_state and State signatures
Components are packed in bit order into a `(nworld, mj_stateSize)` float array. `PHYSICS` (= QPOS|QVEL|ACT|HISTORY|PLASTIC, no TIME bit) is the MPC snapshot; add `WARMSTART` for a bitwise replay; `INTEGRATION` takes everything. **`get_state` does not check the buffer size.**

The next cell writes the course's running helper module `my_env.py` as it stands after Chapter 8 (`reset_done(m, d, done, randomize=None)`, `nonfinite_reset`, `snapshot`, `restore`). The examples keep the same code inline.

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 8.

Ch 6 added make_worlds. Ch 8 adds reset_done, nonfinite_reset, snapshot and restore.
Later chapters extend this file (Ch 9 describe_overflow, Ch 10 randomize, Ch 14 sensor_slices,
Ch 24 capture_step, Ch 26 obs/reward kernels, Ch 30 the environments).
"""
import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done, randomize=None):
    """Reset the worlds whose done flag is True, randomize them, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). `randomize(m, d, done)`, if given, launches
    the per-episode randomization (initial velocity, targets, ...) of the done worlds. It runs
    after reset_data and before forward, so every derived quantity (xpos, cvel, velocity sensors,
    qfrc_bias, ...) matches the final state. Everything stays on the device, so this can be
    captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).

    Cost: forward runs on every world on every call, even when no world is done, which is about
    one extra forward pass per step. It is unconditional so that there is no host branch to break
    graph capture; skipping it would need a conditional graph node, or reading observations after
    the next step instead.
    """
    mjw.reset_data(m, d, reset=done)
    if randomize is not None:
        randomize(m, d, done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS (+ State.TIME if anything reads d.time) for MPC (it includes HISTORY and
    PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)

In [ ]:
# Example 8.4 - ex04_snapshot_restore.py
"""Example 8.4 - snapshot and restore the state of every world with get_state / set_state.

Two free boxes rest on a plane. After 150 steps we snapshot, run 30 steps (reference), restore,
run the same 30 steps again and compare. A QPOS|QVEL snapshot is "almost" exact; adding the
solver warmstart (State.PHYSICS | State.WARMSTART) makes the replay bitwise identical.
Run: python ex04_snapshot_restore.py [--dump-json PATH]
  --dump-json PATH writes mj_stateSize of every State bit for MOCAP_TARGET and PLAYGROUND
  (Rule D, state-packing widget).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

TWO_BOXES = """
<mujoco model="two_boxes">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="box1" pos="0 0 0.3" euler="10 20 0">
      <freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="box2" pos="0.05 0 0.6" euler="0 15 30">
      <freejoint/><geom type="box" size=".1 .1 .1"/></body>
  </worldbody>
</mujoco>
"""

MOCAP_TARGET = """
<mujoco model="mocap_target">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="target" mocap="true" pos="0 0 0.5">
      <geom type="box" size=".05 .05 .05" contype="0" conaffinity="0" rgba="0 1 0 .3"/>
    </body>
    <body name="box" pos="0 0 0.5">
      <freejoint/>
      <geom name="box" type="box" size=".05 .05 .05" mass="0.5" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
  <equality><weld body1="box" body2="target" solref="0.02 1"/></equality>
</mujoco>
"""

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8


# --- the helpers this chapter adds to my_env.py (same code as my_env_step08.py) ---
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) array."""
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (optionally only where the per-world bool mask `active` is True)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


mjm = mujoco.MjModel.from_xml_string(TWO_BOXES)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
for _ in range(150):  # boxes land and stack: contacts and warmstart are active
    mjw.step(m, d)
print("constraint rows (nefc) in world 0 after 150 steps:", int(d.nefc.numpy()[0]))

S = mjw.State
base = snapshot(m, d, mjm, S.INTEGRATION)    # everything, so each test starts from the same state
for name, sig in [("QPOS | QVEL", S.QPOS | S.QVEL),
                  ("PHYSICS", S.PHYSICS),
                  ("PHYSICS | WARMSTART", S.PHYSICS | S.WARMSTART),
                  ("TIME|PHYSICS|WARMSTART", S.TIME | S.PHYSICS | S.WARMSTART),
                  ("INTEGRATION", S.INTEGRATION)]:
    restore(m, d, mjm, base, S.INTEGRATION)
    buf = snapshot(m, d, mjm, sig)            # 1. snapshot
    for _ in range(30):                       # 2. reference rollout
        mjw.step(m, d)
    ref_q, ref_t = d.qpos.numpy().copy(), d.time.numpy().copy()
    restore(m, d, mjm, buf, sig)              # 3. restore (warmstart/time stay at the end values
    for _ in range(30):                       #    unless sig contains them) and replay
        mjw.step(m, d)
    dq = np.abs(d.qpos.numpy() - ref_q).max()
    same_t = bool(np.array_equal(d.time.numpy(), ref_t))
    print(f"{name:22s} size {mujoco.mj_stateSize(mjm, int(sig)):3d}  max |dqpos| = {dq:.3g}"
          f"  time replayed exactly: {same_t}")

print("State.PHYSICS =", int(S.PHYSICS), "includes TIME?", bool(S.PHYSICS & S.TIME))
print("State.FULLPHYSICS includes TIME?", bool(S.FULLPHYSICS & S.TIME))
try:
    snapshot(m, d, mjm, S.QPOS, buf=wp.zeros((NWORLD, 3), dtype=float))
except AssertionError as e:
    print("undersized buffer caught:", e)
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    bits = ["TIME", "QPOS", "QVEL", "ACT", "HISTORY", "WARMSTART", "CTRL", "QFRC_APPLIED",
            "XFRC_APPLIED", "EQ_ACTIVE", "MOCAP_POS", "MOCAP_QUAT", "USERDATA", "PLUGIN", "PLASTIC"]
    out = {"bits": {b: int(getattr(mujoco.mjtState, "mjSTATE_" + b)) for b in bits},
           "composites": {c: int(getattr(S, c)) for c in
                          ["PHYSICS", "FULLPHYSICS", "USER", "INTEGRATION"]},
           "models": {}}
    for mname, xml in [("MOCAP_TARGET", MOCAP_TARGET), ("PLAYGROUND", PLAYGROUND)]:
        mm = mujoco.MjModel.from_xml_string(xml)
        out["models"][mname] = {b: mujoco.mj_stateSize(mm, out["bits"][b]) for b in bits}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 5. Mocap bodies: one target per world
`d.mocap_pos` is `(nworld, nmocap)` of `vec3`; kinematics copies it into the mocap body's pose. Here a kernel moves every world's target on its own circle and a weld drags a box along.

In [ ]:
# Example 8.5 - ex05_mocap_targets.py
"""Example 8.5 - per-world mocap targets: a kernel moves each world's target on its own circle.

MOCAP_TARGET: a mocap body "target" and a free box welded to it. Every step a Warp kernel
writes d.mocap_pos[world, 0] (radius grows with the world index); the weld drags the box along.
Run: python ex05_mocap_targets.py [--dump-json PATH]
  --dump-json PATH writes decimated target and box positions per world (Rule D, mocap replay).
"""
import argparse
import json
import math

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

MOCAP_TARGET = """
<mujoco model="mocap_target">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="target" mocap="true" pos="0 0 0.5">
      <geom type="box" size=".05 .05 .05" contype="0" conaffinity="0" rgba="0 1 0 .3"/>
    </body>
    <body name="box" pos="0 0 0.5">
      <freejoint/>
      <geom name="box" type="box" size=".05 .05 .05" mass="0.5" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
  <equality><weld body1="box" body2="target" solref="0.02 1"/></equality>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
T_END = 2.0  # seconds


@wp.kernel
def circle_targets(time_in: wp.array[float], r_min: float, r_step: float, omega: float,
                   mocap_pos_out: wp.array2d[wp.vec3]):
    worldid = wp.tid()                       # one thread per world
    r = r_min + r_step * float(worldid % 8)  # radius depends on the world (8 distinct radii)
    t = time_in[worldid]
    mocap_pos_out[worldid, 0] = wp.vec3(r * wp.cos(omega * t), r * wp.sin(omega * t), 0.5)


mjm = mujoco.MjModel.from_xml_string(MOCAP_TARGET)
print("nmocap:", mjm.nmocap, "| neq:", mjm.neq, "| nq:", mjm.nq, "| nv:", mjm.nv)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
print("d.mocap_pos shape:", d.mocap_pos.shape, "| d.mocap_quat shape:", d.mocap_quat.shape)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
target = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "target")

omega = 2 * math.pi / T_END  # one lap in T_END seconds
nstep = int(round(T_END / mjm.opt.timestep))
frames = []
for i in range(nstep):
    wp.launch(circle_targets, dim=d.nworld, inputs=[d.time, 0.05, 0.05, omega],
              outputs=[d.mocap_pos])
    mjw.step(m, d)
    if args.dump_json and i % 8 == 0:  # decimate (host read: demo only)
        xp = d.xpos.numpy()[:8]
        frames.append([[round(float(v), 3) for v in (*xp[w, target, :2], *xp[w, box, [0, 2]],
                                                     xp[w, box, 1])] for w in range(8)])
wp.synchronize()

xpos = d.xpos.numpy()
dist = np.linalg.norm(xpos[:, box] - xpos[:, target], axis=1)
r = 0.05 + 0.05 * (np.arange(NWORLD) % 8)
print(f"after {nstep} steps ({T_END:.1f} s):")
print("target radius   [m] =", r[:8])
print("target speed  [m/s] =", r[:8] * omega)
print("box-target dist [m] =", dist[:8])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": 8, "dt": 8 * float(mjm.opt.timestep), "kind": "mocap",
                   "labels": ["target x", "target y", "box x", "box z", "box y"],
                   "radius": [round(float(v), 3) for v in r[:8]], "frames": frames},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 6. Switching equality constraints per world
`d.eq_active` is `(nworld, neq)` bool. An inactive constraint produces no efc rows in that world.

In [ ]:
# Example 8.6 - ex06_eq_toggle.py
"""Example 8.6 - switch an equality constraint off in some worlds with d.eq_active.

MOCAP_TARGET again: the weld holds the box at the target. Odd worlds turn the weld off, so their
box falls to the floor; even worlds keep it. Equality rows (d.ne) appear only where it is on.
Run: python ex06_eq_toggle.py [--dump-json PATH]
  --dump-json PATH writes decimated box heights per world (Rule D, mocap replay, odd tiles).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

MOCAP_TARGET = """
<mujoco model="mocap_target">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="target" mocap="true" pos="0 0 0.5">
      <geom type="box" size=".05 .05 .05" contype="0" conaffinity="0" rgba="0 1 0 .3"/>
    </body>
    <body name="box" pos="0 0 0.5">
      <freejoint/>
      <geom name="box" type="box" size=".05 .05 .05" mass="0.5" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
  <equality><weld body1="box" body2="target" solref="0.02 1"/></equality>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 100

mjm = mujoco.MjModel.from_xml_string(MOCAP_TARGET)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
print("m.eq_active0:", m.eq_active0.numpy(), "| d.eq_active shape:", d.eq_active.shape)

active = np.ones((NWORLD, mjm.neq), dtype=bool)
active[1::2, 0] = False                  # odd worlds: weld off
d.eq_active.assign(active)               # in place: same array object (graph-safe)

frames = []
for i in range(NSTEP):
    mjw.step(m, d)
    if i == 0:
        print("after 1 step:    ne   =", d.ne.numpy()[:8], "| nefc =", d.nefc.numpy()[:8])
    if args.dump_json and i % 2 == 0:
        frames.append([round(float(z), 3) for z in d.qpos.numpy()[:8, 2]])

print(f"after {NSTEP} steps: box z =", d.qpos.numpy()[:8, 2])
print(f"after {NSTEP} steps: ne   =", d.ne.numpy()[:8], "| nefc =", d.nefc.numpy()[:8])
print("contacts per world (odd worlds sit on the floor):",
      np.bincount(d.contact.worldid.numpy()[:int(d.nacon.numpy()[0])], minlength=8)[:8])

mjw.reset_data(m, d)                     # a reset restores eq_active from m.eq_active0
print("after reset_data: eq_active =", d.eq_active.numpy()[:8, 0].astype(int))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": 8, "dt": 2 * float(mjm.opt.timestep), "kind": "height",
                   "zmin": 0.0, "zmax": 0.6, "eq_active": [int(a) for a in active[:8, 0]],
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 7. Control in the loop: step1, policy, step2
`step1` computes positions, velocities and their sensors; your kernel writes `d.ctrl`; `step2` does actuation, acceleration, solve and integration. **With RK4, step2 integrates with Euler.**

In [ ]:
# Example 8.7 - ex07_step1_step2_pd.py
"""Example 8.7 - control in the loop: step1, a PD controller kernel, step2.

PENDULUM: a PD law on hinge1 reads the jointpos/jointvel sensors that step1 just computed and
writes d.ctrl; step2 finishes the step. Each world tracks a different target angle.
Run: python ex07_step1_step2_pd.py [--dump-json PATH]
  --dump-json PATH writes the hinge1 angle of 8 worlds every 5 steps (Rule D, tracking plot).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 300  # 3 s
KP, KD = 50.0, 5.0


@wp.kernel
def pd_control(sensordata_in: wp.array2d[float], target_in: wp.array[float], kp: float,
               kd: float, ctrl_out: wp.array2d[float]):
    worldid = wp.tid()
    q = sensordata_in[worldid, 0]   # sensor "q1" (adr 0)
    qd = sensordata_in[worldid, 1]  # sensor "v1" (adr 1)
    ctrl_out[worldid, 0] = kp * (target_in[worldid] - q) - kd * qd


mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
print("sensor adr:", mjm.sensor_adr, "| integrator:", mjw.IntegratorType(m.opt.integrator).name)
target = wp.array(0.1 + 0.3 * (np.arange(NWORLD) % 8) / 7, dtype=float)  # 0.1 .. 0.4 rad
tgt = target.numpy()


def run(split):
    mjw.reset_data(m, d)
    errs = []
    for i in range(NSTEP):
        if split:
            mjw.step1(m, d)          # positions, velocities, sensors (pos + vel) are fresh
            wp.launch(pd_control, dim=d.nworld, inputs=[d.sensordata, target, KP, KD],
                      outputs=[d.ctrl])
            mjw.step2(m, d)          # actuation, acceleration, solve, integrate
        else:
            # plain step: the controller sees the sensors of the previous step
            wp.launch(pd_control, dim=d.nworld, inputs=[d.sensordata, target, KP, KD],
                      outputs=[d.ctrl])
            mjw.step(m, d)
        errs.append(tgt - d.qpos.numpy()[:, 0])  # host read every step: demo only
    return np.array(errs)


e_split = run(split=True)
print("tracking error q1* - q1 [rad], step1/step2, worlds 0, 3, 7:")
for k in range(49, NSTEP, 50):
    print(f"  t = {0.01 * (k + 1):.1f} s  {e_split[k][[0, 3, 7]]}")
# link2 swings freely (no damping), so the error oscillates around the static value
# g * (m1 * 0.25 + m2 * 0.5) * sin(q*) / kp  (link2 hanging straight down).
print("mean error over the last 1 s:", e_split[-100:].mean(axis=0)[[0, 3, 7]])
print("static estimate g*0.75*sin(q*)/kp: ", np.round(9.81 * 0.75 * np.sin(tgt[[0, 3, 7]]) / KP, 4))
if args.dump_json:
    q = tgt[None, :8] - e_split[::5, :8]
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": 8, "dt": 5 * float(mjm.opt.timestep),
                   "target": np.round(tgt[:8], 3).tolist(), "frames": np.round(q, 3).tolist()},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(q), "frames")
e_plain = run(split=False)
print("max |error| difference, step1/step2 vs step with 1-step-old sensors at 3 s:",
      f"{np.abs(e_split[-1] - e_plain[-1]).max():.2e}")

# RK4 caveat: step2 always integrates with Euler (forward.py: "RK4 defaults to Euler")
mjw.reset_data_keyframe(m, d, 0)
s0 = wp.zeros((NWORLD, mujoco.mj_stateSize(mjm, int(mjw.State.INTEGRATION))), dtype=float)
mjw.get_state(m, d, s0, int(mjw.State.INTEGRATION))
res = {}
for name, integ, split in [("euler step", mjw.IntegratorType.EULER, False),
                           ("rk4 step", mjw.IntegratorType.RK4, False),
                           ("rk4 step1+step2", mjw.IntegratorType.RK4, True)]:
    m.opt.integrator = integ
    mjw.set_state(m, d, s0, int(mjw.State.INTEGRATION))
    for _ in range(20):
        if split:
            mjw.step1(m, d)
            mjw.step2(m, d)
        else:
            mjw.step(m, d)
    res[name] = d.qpos.numpy()[0].copy()
d_rk4 = np.abs(res["rk4 step"] - res["rk4 step1+step2"]).max()
d_eul = np.abs(res["euler step"] - res["rk4 step1+step2"]).max()
print(f"RK4: |step - step1/step2|      = {d_rk4:.2e}")
print(f"RK4 step1/step2 vs Euler step  = {d_eul:.2e}")

## 8. ctrlrange clamps on read
The clamp happens inside `_actuator_force` on a local copy; `d.ctrl` keeps the raw value.

In [ ]:
# Example 8.8 - ex08_ctrl_clamp.py
"""Example 8.8 - ctrlrange is applied when the actuator reads ctrl; d.ctrl itself is never changed.

PENDULUM's motor has ctrlrange="-5 5". We write ctrl values outside the range in some worlds,
run forward, and compare d.ctrl with d.actuator_force (motor: gain 1, no bias, so force = clamped
ctrl; the gear is applied by the transmission and shows up only in d.qfrc_actuator).
Run: python ex08_ctrl_clamp.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

NWORLD = 4

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
print("ctrllimited:", mjm.actuator_ctrllimited, "| ctrlrange:", mjm.actuator_ctrlrange)

d.ctrl.assign(np.array([[-10.0], [2.0], [5.0], [10.0]], dtype=np.float32))
mjw.forward(m, d)
print("d.ctrl            =", d.ctrl.numpy()[:, 0], " <- untouched")
print("d.actuator_force  =", d.actuator_force.numpy()[:, 0], " <- clamped inside _actuator_force")

mjw.step(m, d)
print("after a step, d.ctrl =", d.ctrl.numpy()[:, 0])

m.opt.disableflags |= mjw.DisableBit.CLAMPCTRL   # runtime flag: passed to the kernel as an int
mjw.forward(m, d)
print("CLAMPCTRL disabled: actuator_force =", d.actuator_force.numpy()[:, 0])

## 9. Callbacks: m.callback.control
A Python function `fn(m, d)` that launches kernels. It runs after the velocity sensors and before actuation, inside `forward`/`step`/`step1`. In a captured CUDA graph it runs once, at capture. Under RK4, `mjw.step` calls it 4 times per step; stages 2-4 see intermediate `qpos`/`qvel` and stage-1 `sensordata`.

In [ ]:
# Example 8.9 - ex09_control_callback.py
"""Example 8.9 - put the controller inside mjw.step with m.callback.control.

The callback is a plain Python function (m, d) -> None that launches a kernel writing d.ctrl.
MJWarp calls it in _forward right after the velocity sensors and before actuation, so a plain
mjw.step loop (or a captured CUDA graph of it) runs the controller every step.
Run: python ex09_control_callback.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 300  # 3 s


@wp.kernel
def pd_control(sensordata_in: wp.array2d[float], target: float, kp: float, kd: float,
               ctrl_out: wp.array2d[float]):
    worldid = wp.tid()
    q = sensordata_in[worldid, 0]   # sensor "q1"
    qd = sensordata_in[worldid, 1]  # sensor "v1"
    ctrl_out[worldid, 0] = kp * (target - q) - kd * qd


calls = [0]


def control(m, d):
    """Called by MJWarp as m.callback.control(m, d): once per forward / step1 / Euler or implicit
    step, but 4 times per RK4 step (stages 2-4 see intermediate qpos/qvel and stale sensors)."""
    calls[0] += 1  # Python side effect: runs once per call here, but only ONCE at graph capture
    wp.launch(pd_control, dim=d.nworld, inputs=[d.sensordata, 0.3, 50.0, 5.0], outputs=[d.ctrl])


mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
print("default callbacks:", m.callback)

for use_cb in (False, True):
    m.callback.control = control if use_cb else None
    mjw.reset_data_keyframe(m, d, 0)        # start horizontal: hinge1 = pi/2
    for _ in range(NSTEP):
        mjw.step(m, d)
    label = "with callback   " if use_cb else "without callback"
    print(f"{label}: hinge1 after {NSTEP} steps = {d.qpos.numpy()[:4, 0]}  (target 0.3)")
print("callback calls:", calls[0], "for", NSTEP, "steps")

m.opt.disableflags |= mjw.DisableBit.ACTUATION  # with actuation disabled the callback is skipped
mjw.step(m, d)
print("calls after one step with ACTUATION disabled:", calls[0])

## 10. Device-side auto-reset with a NaN guard
Done kernel, NaN/Inf guard (MJWarp has no `mj_checkPos`), masked reset, re-randomize, then forward (randomize before forward, so the observations match the randomized state). This example takes about 20 s on a CPU; on a GPU it runs 4096 worlds.

In [ ]:
# Example 8.10 - ex10_auto_reset.py
"""Example 8.10 - device-side auto-reset: a done kernel, a masked reset, and a non-finite guard.

BALL_DROP with a random initial velocity per episode. An episode ends when the ball lands
(z < 0.15), when it lasts longer than 0.8 s, or when qpos/qvel is not finite (MJWarp has no
mj_checkPos/mj_checkVel). Done worlds are reset, re-randomized and recomputed on the device every step;
no host read is needed inside the loop. World 1 gets a NaN injected once to test the guard.
Run: python ex10_auto_reset.py [--dump-json PATH]
  --dump-json PATH writes decimated ball heights and done flags of 8 worlds (Rule D, timeline).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 5000 if ON_GPU else 1000   # 10 s / 2 s of simulated time


# --- the helpers this chapter adds to my_env.py (same code as my_env_step08.py) ---
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done, randomize=None):
    """Reset the worlds whose done flag is True, randomize them, then recompute derived quantities."""
    mjw.reset_data(m, d, reset=done)      # bool mask of shape (nworld,): no host sync
    if randomize is not None:
        randomize(m, d, done)             # before forward, so forward sees the final state
    mjw.forward(m, d)                     # reset does not recompute xpos, sensors, ...
    # forward runs on ALL worlds, every call, even when no world is done: about one extra pipeline
    # pass per step. Unconditional on purpose: no host branch, so the loop stays graph-capturable.


def nonfinite_reset(m, d, done):
    """Mark worlds with NaN/Inf in qpos or qvel as done (in place, OR-ed into `done`)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- task-specific kernels ---
@wp.kernel
def done_kernel(qpos_in: wp.array2d[float], time_in: wp.array[float], z_min: float, t_max: float,
                done_out: wp.array[bool], reason_out: wp.array2d[int]):
    worldid = wp.tid()
    landed = qpos_in[worldid, 2] < z_min
    timeout = time_in[worldid] > t_max
    done_out[worldid] = landed or timeout
    if landed:
        reason_out[worldid, 0] += 1
    elif timeout:
        reason_out[worldid, 1] += 1


@wp.kernel
def count_nonfinite(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                    reason_out: wp.array2d[int]):
    worldid = wp.tid()
    if not (wp.isfinite(qpos_in[worldid, 2]) and wp.isfinite(qvel_in[worldid, 2])):
        reason_out[worldid, 2] += 1


@wp.kernel
def randomize_velocity(done_in: wp.array[bool], seed: int, episode_in: wp.array[int],
                       qvel_out: wp.array2d[float]):
    worldid = wp.tid()
    if not done_in[worldid]:
        return
    episode_in[worldid] += 1
    rng = wp.rand_init(seed, worldid * 100003 + episode_in[worldid])
    qvel_out[worldid, 0] = wp.randf(rng, -1.0, 1.0)  # vx
    qvel_out[worldid, 1] = wp.randf(rng, -1.0, 1.0)  # vy
    qvel_out[worldid, 2] = wp.randf(rng, 0.0, 4.0)   # vz (up)


mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

done = wp.ones(NWORLD, dtype=bool)          # allocate everything before the loop
episode = wp.zeros(NWORLD, dtype=int)       # episode counter, also the RNG stream id
reason = wp.zeros((NWORLD, 3), dtype=int)   # landed, timeout, non-finite


def randomize(m, d, done):
    """Per-episode randomization of the done worlds (passed to reset_done)."""
    wp.launch(randomize_velocity, dim=NWORLD, inputs=[done, 42, episode], outputs=[d.qvel])


randomize(m, d, done)                       # first episode of every world

frames = []
done_since = np.zeros(8, dtype=bool)
for i in range(NSTEP):
    mjw.step(m, d)
    if i == 300:                                  # inject a blow-up in world 1 (test only)
        q = d.qvel.numpy()
        q[1, 2] = np.nan
        d.qvel.assign(q)
    wp.launch(done_kernel, dim=NWORLD, inputs=[d.qpos, d.time, 0.15, 0.8],
              outputs=[done, reason])
    wp.launch(count_nonfinite, dim=NWORLD, inputs=[d.qpos, d.qvel], outputs=[reason])
    nonfinite_reset(m, d, done)                   # NaN worlds join the done mask
    reset_done(m, d, done, randomize)             # masked reset, re-randomize, forward
    if args.dump_json:                            # host reads for the figure only
        done_since |= done.numpy()[:8]            # remember resets between sampled frames
        if i % 10 == 9:
            z = d.qpos.numpy()[:8, 2]
            frames.append([[round(float(z[w]), 3), int(done_since[w])] for w in range(8)])
            done_since[:] = False
wp.synchronize()

r = reason.numpy()
print(f"{NSTEP} steps = {NSTEP * mjm.opt.timestep:.1f} s per world")
print("episodes finished per world:", r[:8].sum(axis=1))
print("  landed  (z < 0.15):        ", r[:8, 0])
print("  timeout (t > 0.8 s):       ", r[:8, 1])
print("  non-finite (NaN guard):    ", r[:8, 2])
print("all worlds finite now:", bool(np.isfinite(d.qpos.numpy()).all()))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": 8, "dt": 10 * float(mjm.opt.timestep), "kind": "episodes",
                   "labels": ["ball z", "done in the last 10 steps"], "nan_world": 1, "nan_step": 300,
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 11. Worlds as parallel rollouts: random-shooting MPC
All worlds are copies of one cartpole; each tries its own random action sequence; the best first action is applied in every world after restoring the snapshot.

In [ ]:
# Example 8.11 - ex11_mpc_rollouts.py
"""Example 8.11 - worlds as parallel rollouts: random-shooting MPC on CARTPOLE.

All NWORLD worlds hold the same state. Each MPC iteration: snapshot, give every world its own
random action sequence, roll out H steps while a kernel sums an "upright" reward per world,
pick the best world on the device, restore the snapshot, apply that world's first action for
one control interval in every world (so the worlds stay identical), repeat.
Run: python ex11_mpc_rollouts.py [--dump-json PATH]
  --dump-json PATH writes, per iteration, the pole angle along 15 sampled rollouts plus the
  chosen one (Rule D, MPC fan figure).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 64   # number of sampled action sequences
HOLD = 5                          # physics steps per control knot (zero-order hold)
KNOTS = 10                        # horizon H = KNOTS * HOLD = 50 steps = 0.5 s
NITER = 20                        # MPC iterations; each applies one knot (5 steps)
THETA0 = 0.4                      # start with the pole tilted 0.4 rad from upright


# --- the helpers this chapter adds to my_env.py (same code as my_env_step08.py) ---
def snapshot(m, d, mjm, sig, buf=None):
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- MPC kernels (all on the device) ---
@wp.kernel
def sample_actions(seed: int, it: int, actions_out: wp.array2d[float]):
    worldid, k = wp.tid()
    rng = wp.rand_init(seed, (it * 65536 + worldid) * 64 + k)
    a = wp.randf(rng, -1.0, 1.0)
    if worldid == 0:
        a = 0.0                     # world 0 always tries "do nothing" as a baseline
    actions_out[worldid, k] = a


@wp.kernel
def set_ctrl(actions_in: wp.array2d[float], k: int, ctrl_out: wp.array2d[float]):
    worldid = wp.tid()
    ctrl_out[worldid, 0] = actions_in[worldid, k]


@wp.kernel
def add_reward(qpos_in: wp.array2d[float], ctrl_in: wp.array2d[float],
               score_out: wp.array[float]):
    worldid = wp.tid()
    x = qpos_in[worldid, 0]
    theta = qpos_in[worldid, 1]    # 0 = upright
    u = ctrl_in[worldid, 0]
    score_out[worldid] += wp.cos(theta) - 0.05 * x * x - 0.01 * u * u   # weak cart term, see below


@wp.kernel
def argmax_world(score_in: wp.array[float], nworld: int, best_out: wp.array[int]):
    best = int(0)
    for w in range(1, nworld):      # one thread; fine for a few thousand worlds
        if score_in[w] > score_in[best]:
            best = w
    best_out[0] = best


@wp.kernel
def apply_best(actions_in: wp.array2d[float], best_in: wp.array[int],
               ctrl_out: wp.array2d[float]):
    worldid = wp.tid()
    ctrl_out[worldid, 0] = actions_in[best_in[0], 0]   # same first action in every world


mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 1] = THETA0
d.qpos.assign(qpos)

SIG = mjw.State.TIME | mjw.State.PHYSICS     # PHYSICS has no TIME bit; add it so d.time
buf = snapshot(m, d, mjm, SIG)               # is restored too. Allocated once, reused.
actions = wp.zeros((NWORLD, KNOTS), dtype=float)
score = wp.zeros(NWORLD, dtype=float)
best = wp.zeros(1, dtype=int)
print("state size per world:", buf.shape[1], f"(TIME 1 + nq {mjm.nq} + nv {mjm.nv})")

angles, chosen, fans, xs = [THETA0], [], [], [0.0]
for it in range(NITER):
    snapshot(m, d, mjm, SIG, buf)                                     # 1. snapshot
    wp.launch(sample_actions, dim=(NWORLD, KNOTS), inputs=[7, it], outputs=[actions])
    score.zero_()
    fan = [d.qpos.numpy()[:, 1].copy()] if args.dump_json else []
    for k in range(KNOTS):                                            # 2. roll out H steps
        wp.launch(set_ctrl, dim=NWORLD, inputs=[actions, k], outputs=[d.ctrl])
        for _ in range(HOLD):
            mjw.step(m, d)
            wp.launch(add_reward, dim=NWORLD, inputs=[d.qpos, d.ctrl], outputs=[score])
        if args.dump_json:
            fan.append(d.qpos.numpy()[:, 1].copy())
    wp.launch(argmax_world, dim=1, inputs=[score, NWORLD], outputs=[best])  # 3. choose
    restore(m, d, mjm, buf, SIG)                                      # 4. restore
    wp.launch(apply_best, dim=NWORLD, inputs=[actions, best], outputs=[d.ctrl])
    for _ in range(HOLD):                                             # 5. apply one knot
        mjw.step(m, d)
    # host reads below are for printing only (one small copy per iteration)
    angles.append(float(d.qpos.numpy()[0, 1]))
    xs.append(float(d.qpos.numpy()[0, 0]))
    chosen.append(float(actions.numpy()[best.numpy()[0], 0]))
    if args.dump_json:
        b = int(best.numpy()[0])
        rows = [w for w in range(15) if w != b] + [b]      # 15 sampled rollouts + the best
        fan_np = np.array(fan)                              # (KNOTS + 1, NWORLD)
        fans.append({"rollouts": np.round(fan_np[:, rows].T, 3).tolist(), "best_world": b,
                     "theta": round(angles[-1], 3)})

q = d.qpos.numpy()
t_now = round(float(d.time.numpy()[0]), 3)
print("worlds identical after MPC:", bool(np.all(q == q[0])), "| time:", t_now)
print("chosen first actions:", np.array(chosen[:10]), "...")
print("pole angle every 0.05 s [rad]:")
print(np.array(angles))
# the cost penalises x only weakly (0.05 x^2) over a 0.5 s horizon: the cart drifts toward the
# +-1.8 m slider limit, which it reaches after about 1.8 s if you raise NITER to 40
print(f"cart x after {t_now} s: {xs[-1]:+.3f} m (max |x| {max(abs(v) for v in xs):.3f} m, limit 1.8 m)")

# baseline: same start, zero force
mjw.reset_data(m, d)
d.qpos.assign(qpos)
for _ in range(NITER * HOLD):
    mjw.step(m, d)
print(f"without control after {NITER * HOLD * mjm.opt.timestep:.1f} s: pole angle =",
      round(float(d.qpos.numpy()[0, 1]), 3))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "hold": HOLD, "knots": KNOTS,
                   "dt": float(mjm.opt.timestep), "theta": [round(a, 3) for a in angles],
                   "iters": fans}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(fans), "iterations")

### Interactive MPC demo
Use the form fields (or edit the values), re-run, and watch the video of the executed trajectory. Example 8.11 stops at 1 s, just before its cart reaches the slider limit (about 1.8 s); this demo plans 1 s ahead and also penalises the cart position and velocity, so it keeps the cart on the rail. With 64 samples on a CPU it holds the pole within about 0.3 rad for starts up to about 0.4 rad; larger tilts can still fail. On a CPU runtime the renderer is usually unavailable; the cell then plots the angle and the cart position.

In [ ]:
# Interactive MPC demo. It reuses CARTPOLE, the MPC kernels and snapshot/restore from the
# Example 8.11 cell, but builds its own model, data and buffers (names end in _mpc), so it still
# works after the exercise cells below have redefined mjm, m, d and NWORLD.
# It changes two things against Example 8.11, which lets the cart drift into the +-1.8 m slider
# limit after about 1.8 s: a 1 s horizon (KNOTS_MPC = 20) and a reward that also penalises the
# cart position and velocity, so the planner sees the rail end coming.
THETA_START = 0.4   #@param {type:"slider", min:-0.5, max:0.5, step:0.05}
NITER_DEMO = 40     #@param {type:"integer"}
SEED_DEMO = 11      #@param {type:"integer"}
HOLD_MPC, KNOTS_MPC = 5, 20                  # horizon 20 knots x 5 steps = 1 s
NWORLD_MPC = 1024 if ON_GPU else 64          # sampled action sequences per iteration


@wp.kernel
def add_reward_cart(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                    ctrl_in: wp.array2d[float], score_out: wp.array[float]):
    worldid = wp.tid()
    x = qpos_in[worldid, 0]
    xdot = qvel_in[worldid, 0]
    u = ctrl_in[worldid, 0]
    score_out[worldid] += (wp.cos(qpos_in[worldid, 1]) - 0.2 * x * x - 0.05 * xdot * xdot
                           - 0.01 * u * u)


mjm_mpc = mujoco.MjModel.from_xml_string(CARTPOLE)
m_mpc = mjw.put_model(mjm_mpc)
d_mpc = mjw.make_data(mjm_mpc, nworld=NWORLD_MPC)
qpos_demo = np.tile(mjm_mpc.qpos0, (NWORLD_MPC, 1))
qpos_demo[:, 1] = THETA_START
d_mpc.qpos.assign(qpos_demo)
SIG_MPC = mjw.State.TIME | mjw.State.PHYSICS
buf_mpc = snapshot(m_mpc, d_mpc, mjm_mpc, SIG_MPC)
actions_mpc = wp.zeros((NWORLD_MPC, KNOTS_MPC), dtype=float)
score_mpc = wp.zeros(NWORLD_MPC, dtype=float)
best_mpc = wp.zeros(1, dtype=int)

traj = []
for it in range(NITER_DEMO):
    snapshot(m_mpc, d_mpc, mjm_mpc, SIG_MPC, buf_mpc)
    wp.launch(sample_actions, dim=(NWORLD_MPC, KNOTS_MPC), inputs=[SEED_DEMO, it], outputs=[actions_mpc])
    score_mpc.zero_()
    for k in range(KNOTS_MPC):
        wp.launch(set_ctrl, dim=NWORLD_MPC, inputs=[actions_mpc, k], outputs=[d_mpc.ctrl])
        for _ in range(HOLD_MPC):
            mjw.step(m_mpc, d_mpc)
            wp.launch(add_reward_cart, dim=NWORLD_MPC, inputs=[d_mpc.qpos, d_mpc.qvel, d_mpc.ctrl],
                      outputs=[score_mpc])
    wp.launch(argmax_world, dim=1, inputs=[score_mpc, NWORLD_MPC], outputs=[best_mpc])
    restore(m_mpc, d_mpc, mjm_mpc, buf_mpc, SIG_MPC)
    wp.launch(apply_best, dim=NWORLD_MPC, inputs=[actions_mpc, best_mpc], outputs=[d_mpc.ctrl])
    for _ in range(HOLD_MPC):
        mjw.step(m_mpc, d_mpc)
        traj.append(d_mpc.qpos.numpy()[0].copy())   # host read for the video only
traj = np.array(traj)
max_x = float(np.abs(traj[:, 0]).max())
print(f"start {THETA_START:+.2f} rad -> final pole angle {traj[-1, 1]:+.3f} rad, "
      f"cart x {traj[-1, 0]:+.3f} m, max |x| {max_x:.3f} m")
if max_x >= 1.79:
    print("WARNING: the cart reached the +-1.8 m slider limit; the planner lost track of the rail. "
          "Try a smaller THETA_START, another SEED_DEMO, or more worlds.")

# Render world 0 with the C renderer (needs EGL; the ICD cell at the top prepared it on Colab).
# Falls back to a plot of the pole angle.
try:
    import mediapy as media
    renderer = mujoco.Renderer(mjm_mpc, height=240, width=320)
    mjd = mujoco.MjData(mjm_mpc)
    video = []
    for q in traj[::2]:
        mjd.qpos[:] = q
        mujoco.mj_forward(mjm_mpc, mjd)
        renderer.update_scene(mjd, camera="front")
        video.append(renderer.render())
    media.show_video(video, fps=round(1 / (2 * mjm_mpc.opt.timestep)))
except Exception as e:
    print(f"skipped video ({type(e).__name__}: {e}); plotting the pole angle instead")
    try:
        import matplotlib.pyplot as plt
        t = np.arange(len(traj)) * mjm_mpc.opt.timestep
        plt.plot(t, traj[:, 1], label="pole angle [rad]")
        plt.plot(t, traj[:, 0], label="cart x [m]")
        plt.xlabel("t [s]"); plt.axhline(0, color="gray", lw=0.5); plt.legend()
        plt.show()
    except ImportError:
        print("skipped: matplotlib not installed")

## Exercise 8.1: reset only the worlds whose domino tipped over
Write a kernel that fills the bool mask `tipped` on the device (tipped when `xmat[w, body][2, 2] < cos(45 deg)`), then reset exactly those worlds. Run the cell: the asserts check your work.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

DOMINO = """
<mujoco model="domino">
  <option timestep="0.005"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="domino" pos="0 0 0.2"><freejoint/><geom type="box" size=".05 .05 .2"/></body>
  </worldbody>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(DOMINO)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
tilt = np.linspace(0.0, 0.5, NWORLD)            # initial tilt about y [rad]
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 3] = np.cos(tilt / 2)
qpos[:, 5] = np.sin(tilt / 2)
qpos[:, 2] = 0.2 * np.cos(tilt) + 0.05 * np.sin(tilt) + 0.001
d.qpos.assign(qpos)
for _ in range(300):
    mjw.step(m, d)
body = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "domino")
expect = d.xmat.numpy()[:, body, 2, 2] < np.cos(np.pi / 4)   # read BEFORE the reset

tipped = wp.zeros(NWORLD, dtype=bool)
# ---- your code: launch a kernel that fills `tipped`, then reset those worlds ----

# ---- checks ----
assert expect.any() and not expect.all()
assert np.array_equal(tipped.numpy(), expect)
assert np.all(d.time.numpy()[expect] == 0.0) and np.all(d.time.numpy()[~expect] > 0.0)
print("ok")

In [ ]:
#@title Exercise 8.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

DOMINO = """
<mujoco model="domino">
  <option timestep="0.005"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="domino" pos="0 0 0.2"><freejoint/><geom type="box" size=".05 .05 .2"/></body>
  </worldbody>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(DOMINO)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
tilt = np.linspace(0.0, 0.5, NWORLD)            # initial tilt about y [rad]
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 3] = np.cos(tilt / 2)
qpos[:, 5] = np.sin(tilt / 2)
qpos[:, 2] = 0.2 * np.cos(tilt) + 0.05 * np.sin(tilt) + 0.001
d.qpos.assign(qpos)
for _ in range(300):
    mjw.step(m, d)
body = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "domino")
expect = d.xmat.numpy()[:, body, 2, 2] < np.cos(np.pi / 4)   # read BEFORE the reset

tipped = wp.zeros(NWORLD, dtype=bool)
@wp.kernel
def tipped_kernel(xmat_in: wp.array2d[wp.mat33], body: int, cos_max: float,
                  tipped_out: wp.array[bool]):
    worldid = wp.tid()
    tipped_out[worldid] = xmat_in[worldid, body][2, 2] < cos_max   # body z . world z


wp.launch(tipped_kernel, dim=NWORLD, inputs=[d.xmat, body, np.cos(np.pi / 4)], outputs=[tipped])
mjw.reset_data(m, d, reset=tipped)

# ---- checks ----
assert expect.any() and not expect.all()
assert np.array_equal(tipped.numpy(), expect)
assert np.all(d.time.numpy()[expect] == 0.0) and np.all(d.time.numpy()[~expect] > 0.0)
print("ok")

## Exercise 8.2: attach the weld when the box is near the target
Write a kernel that sets `eq_active[w, 0] = True` when the box is closer than 5 cm to the target.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

MOCAP_TARGET = """
<mujoco model="mocap_target">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="target" mocap="true" pos="0 0 0.5">
      <geom type="box" size=".05 .05 .05" contype="0" conaffinity="0" rgba="0 1 0 .3"/>
    </body>
    <body name="box" pos="0 0 0.5">
      <freejoint/>
      <geom name="box" type="box" size=".05 .05 .05" mass="0.5" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
  <equality><weld body1="box" body2="target" solref="0.02 1"/></equality>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(MOCAP_TARGET)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
d.eq_active.assign(np.zeros((NWORLD, 1), dtype=bool))   # start detached: the box falls
for _ in range(100):
    mjw.step(m, d)                                        # box now rests on the floor (z = 0.05)
offset = np.linspace(0.0, 0.105, NWORLD)                  # target 0 .. 10.5 cm from the box
tgt = np.zeros((NWORLD, 1, 3), dtype=np.float32)
tgt[:, 0, 0] = d.qpos.numpy()[:, 0] + offset
tgt[:, 0, 1] = d.qpos.numpy()[:, 1]
tgt[:, 0, 2] = 0.05
d.mocap_pos.assign(tgt)
mjw.forward(m, d)                                         # fresh xpos for the check
# ---- your code: a kernel that sets eq_active[w, 0] = True when |box - target| < 0.05 ----

# ---- checks ----
dist = np.linalg.norm(d.xpos.numpy()[:, box] - d.mocap_pos.numpy()[:, 0], axis=1)
assert np.array_equal(d.eq_active.numpy()[:, 0], dist < 0.05)
tgt[:, 0, 2] = 0.4                                        # lift every target
d.mocap_pos.assign(tgt)
for _ in range(200):
    mjw.step(m, d)
z = d.qpos.numpy()[:, 2]
assert np.all(z[dist < 0.05] > 0.3) and np.all(z[dist >= 0.05] < 0.06)
print("ok")

In [ ]:
#@title Exercise 8.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

MOCAP_TARGET = """
<mujoco model="mocap_target">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="target" mocap="true" pos="0 0 0.5">
      <geom type="box" size=".05 .05 .05" contype="0" conaffinity="0" rgba="0 1 0 .3"/>
    </body>
    <body name="box" pos="0 0 0.5">
      <freejoint/>
      <geom name="box" type="box" size=".05 .05 .05" mass="0.5" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
  <equality><weld body1="box" body2="target" solref="0.02 1"/></equality>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(MOCAP_TARGET)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
d.eq_active.assign(np.zeros((NWORLD, 1), dtype=bool))   # start detached: the box falls
for _ in range(100):
    mjw.step(m, d)                                        # box now rests on the floor (z = 0.05)
offset = np.linspace(0.0, 0.105, NWORLD)                  # target 0 .. 10.5 cm from the box
tgt = np.zeros((NWORLD, 1, 3), dtype=np.float32)
tgt[:, 0, 0] = d.qpos.numpy()[:, 0] + offset
tgt[:, 0, 1] = d.qpos.numpy()[:, 1]
tgt[:, 0, 2] = 0.05
d.mocap_pos.assign(tgt)
mjw.forward(m, d)                                         # fresh xpos for the check
@wp.kernel
def attach_kernel(xpos_in: wp.array2d[wp.vec3], mocap_pos_in: wp.array2d[wp.vec3], body: int,
                  radius: float, eq_active_out: wp.array2d[bool]):
    worldid = wp.tid()
    if wp.length(xpos_in[worldid, body] - mocap_pos_in[worldid, 0]) < radius:
        eq_active_out[worldid, 0] = True


wp.launch(attach_kernel, dim=NWORLD, inputs=[d.xpos, d.mocap_pos, box, 0.05],
          outputs=[d.eq_active])

# ---- checks ----
dist = np.linalg.norm(d.xpos.numpy()[:, box] - d.mocap_pos.numpy()[:, 0], axis=1)
assert np.array_equal(d.eq_active.numpy()[:, 0], dist < 0.05)
tgt[:, 0, 2] = 0.4                                        # lift every target
d.mocap_pos.assign(tgt)
for _ in range(200):
    mjw.step(m, d)
z = d.qpos.numpy()[:, 2]
assert np.all(z[dist < 0.05] > 0.3) and np.all(z[dist >= 0.05] < 0.06)
print("ok")

## Stretch: step1/step2 vs callback, on the GPU
Time (a) `step1` + PD kernel + `step2`, (b) `mjw.step` with `m.callback.control`, (c) version (b) captured in a CUDA graph, at `NWORLD = 4096`. Warm up once, `wp.synchronize()` before reading the clock, report world-steps/s. Check that (a) and (b) give identical `qpos` for the Euler pendulum; then switch to RK4 and explain the difference.

In [ ]:
# your timing code here